# ExU-Trans-style BraTS2020 Reproduction

This notebook maps the ExU-Trans paper to the locally downloaded BraTS2020 Training + Validation dataset and the downloaded Kaggle notebook/code references in this repository.

This is **not** official author code. Where the paper or local materials are underspecified, the implementation below uses a documented approximation that stays practical on local hardware.

In [ ]:
# Optional bootstrap for a fresh environment.
# Uncomment if the required packages are missing.
# !pip install -q torch torchvision nibabel matplotlib scipy scikit-learn tqdm

from __future__ import annotations

import random
from functools import lru_cache
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import nibabel as nib
from scipy.ndimage import binary_erosion, distance_transform_edt
from sklearn.model_selection import train_test_split

plt.style.use('seaborn-v0_8-darkgrid')

In [ ]:
RANDOM_SEED = 42

def set_seed(seed: int = RANDOM_SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

CONFIG = {
    "DATA_ROOT_TRAIN": Path(r"D:\GitHub\aysan\class-projects\1\dataset\BraTS2020_TrainingData\MICCAI_BraTS2020_TrainingData"),
    "DATA_ROOT_OFFICIAL_VAL": Path(r"D:\GitHub\aysan\class-projects\1\dataset\BraTS2020_ValidationData\MICCAI_BraTS2020_ValidationData"),
    "TRAIN_RATIO": 0.70,
    "VAL_RATIO": 0.15,
    "TEST_RATIO": 0.15,
    "RANDOM_SEED": RANDOM_SEED,
    "USE_DEBUG_SUBSET": False,
    "DEBUG_NUM_CASES": 2,
    "debug_max_slices_per_case": 2,
    "PATCH_OR_SLICE_MODE": "slice",
    "image_size": 128,
    "patch_size": 16,
    "batch_size": 4,
    "num_workers": 0,
    "label_mode": "whole_tumor",
    "epochs": 100,
    "lr": 1e-4,
    "weight_decay": 1e-4,
    "attr_loss_weight": 0.1,
    "SAVE_OUTPUTS": True,
    "SAVE_CHECKPOINTS": False,
}

assert abs((CONFIG["TRAIN_RATIO"] + CONFIG["VAL_RATIO"] + CONFIG["TEST_RATIO"]) - 1.0) < 1e-6

print("device:", DEVICE)
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

print("\n=== CONFIGURATION ===")
for key, value in sorted(CONFIG.items()):
    print(f"  {key}: {value}")
print("\nDataset split explanation:")
print("  - Labeled training cases (n=369) are split internally: 70% train, 15% val, 15% test")
print("  - Official validation cases (n=125) are treated as unlabeled inference-only")
print(f"  - Mode: {CONFIG['PATCH_OR_SLICE_MODE'].upper()} (2D slice-based, not 3D)")

## Dataset Discovery

BraTS2020 training: 369 cases with masks. Validation: 125 cases (no masks). Uses internal 70/15/15 split.

In [ ]:
def _find_case_dirs(root: Path, prefix: str) -> List[Path]:
    return sorted([p for p in root.iterdir() if p.is_dir() and p.name.startswith(prefix)]) if root.exists() else []

def _case_files(case_dir: Path) -> Dict[str, Optional[Path]]:
    files = {'flair': None, 't1': None, 't1ce': None, 't2': None, 'seg': None}
    for path in case_dir.glob('*.nii*'):
        n = path.name.lower()
        if '_flair' in n:
            files['flair'] = path
        elif n.endswith('_t1.nii') or n.endswith('_t1.nii.gz'):
            files['t1'] = path
        elif '_t1ce' in n or '_t1gd' in n or '_t1g' in n:
            files['t1ce'] = path
        elif '_t2' in n:
            files['t2'] = path
        elif '_seg' in n:
            files['seg'] = path
    return files

@lru_cache(maxsize=16)
def load_nifti(path: str) -> np.ndarray:
    return np.transpose(nib.load(path).get_fdata().astype(np.float32), (2, 0, 1))

def zscore_non_background(volume: np.ndarray) -> np.ndarray:
    mask = volume > 0
    if mask.any():
        mean, std = float(volume[mask].mean()), float(volume[mask].std())
        std = 1.0 if std < 1e-6 else std
        volume = (volume - mean) / std
    else:
        volume = volume - volume.mean()
    return volume.astype(np.float32)

def load_case(case_dir: Path) -> Dict[str, object]:
    files = _case_files(case_dir)
    modalities = {}
    for key in ('flair', 't1', 't1ce', 't2'):
        if files[key] is None:
            raise FileNotFoundError('missing ' + key)
        modalities[key] = zscore_non_background(load_nifti(str(files[key])))
    mask = load_nifti(str(files['seg'])).astype(np.int64) if files['seg'] is not None else None
    return {'case_dir': case_dir, 'modalities': modalities, 'mask': mask, 'files': files}

def make_binary_whole_tumor(mask: np.ndarray) -> np.ndarray:
    return (mask > 0).astype(np.float32)

def make_subregion_classes(mask: np.ndarray) -> np.ndarray:
    remapped = np.zeros_like(mask, dtype=np.int64)
    remapped[mask == 1] = 1
    remapped[mask == 2] = 2
    remapped[mask == 4] = 3
    return remapped

def extract_tumor_regions(seg_mask: np.ndarray) -> Dict[str, np.ndarray]:
    wt = ((seg_mask == 1) | (seg_mask == 2) | (seg_mask == 4)).astype(np.float32)
    tc = ((seg_mask == 1) | (seg_mask == 4)).astype(np.float32)
    et = (seg_mask == 4).astype(np.float32)
    return {'WT': wt, 'TC': tc, 'ET': et}

def label_map(mask_2d: np.ndarray, label_mode: str) -> np.ndarray:
    if label_mode == 'whole_tumor':
        return make_binary_whole_tumor(mask_2d)
    else:
        return make_subregion_classes(mask_2d)

def summarize_dataset(config: Dict) -> Dict[str, object]:
    train_root = config['DATA_ROOT_TRAIN']
    official_val_root = config['DATA_ROOT_OFFICIAL_VAL']
    all_train_cases = _find_case_dirs(train_root, 'BraTS20_Training_')
    official_val_cases = _find_case_dirs(official_val_root, 'BraTS20_Validation_')
    if len(all_train_cases) == 0:
        print('[WARNING] No training cases found')
        return {'train_cases': [], 'val_cases': [], 'test_cases': [], 'official_val_cases': [], 'train_count': 0, 'val_count': 0, 'test_count': 0, 'official_val_count': 0}
    train_cases, temp_cases = train_test_split(all_train_cases, train_size=config['TRAIN_RATIO'], random_state=config['RANDOM_SEED'], shuffle=True)
    val_ratio = config['VAL_RATIO'] / (config['VAL_RATIO'] + config['TEST_RATIO'])
    val_cases, test_cases = train_test_split(temp_cases, train_size=val_ratio, random_state=config['RANDOM_SEED'], shuffle=True)
    train_cases = sorted(train_cases)
    val_cases = sorted(val_cases)
    test_cases = sorted(test_cases)
    official_val_cases = sorted(official_val_cases)
    summary = {'train_cases': train_cases, 'val_cases': val_cases, 'test_cases': test_cases, 'official_val_cases': official_val_cases}
    summary['train_count'] = len(train_cases)
    summary['val_count'] = len(val_cases)
    summary['test_count'] = len(test_cases)
    summary['official_val_count'] = len(official_val_cases)
    print('Dataset Summary:')
    print(f'Training: {len(all_train_cases)} split as train={summary["train_count"]} val={summary["val_count"]} test={summary["test_count"]}')
    print(f'Official validation: {summary["official_val_count"]}')
    return summary

summary = summarize_dataset(CONFIG)

In [ ]:
class BratsSliceDataset(Dataset):
    def __init__(self, case_dirs: Sequence[Path], image_size: int, label_mode: str, training: bool, debug_max_slices_per_case: Optional[int] = None, augment: bool = False, allow_missing_seg: bool = False):
        self.image_size = image_size
        self.label_mode = label_mode
        self.training = training
        self.augment = augment
        self.allow_missing_seg = allow_missing_seg
        self.items = []
        for case_dir in case_dirs:
            files = _case_files(case_dir)
            seg_path = files['seg']
            if seg_path is None:
                if allow_missing_seg:
                    self.items.append((case_dir, 0, True))
                continue

            # Only the segmentation mask is needed to decide which slices to index.
            # Full modality volumes stay lazy and are loaded in __getitem__.
            mask = load_nifti(str(seg_path)).astype(np.int64)
            tumor_slices = np.where(mask.reshape(mask.shape[0], -1).sum(axis=1) > 0)[0].tolist()
            selected = tumor_slices if tumor_slices else [mask.shape[0] // 2]
            if debug_max_slices_per_case is not None:
                selected = selected[:debug_max_slices_per_case]
            self.items.extend([(case_dir, s, False) for s in selected])

    def __len__(self):
        return len(self.items)

    def _augment(self, image, mask):
        if random.random() < 0.5:
            image = torch.flip(image, dims=[2]); mask = torch.flip(mask, dims=[1])
        if random.random() < 0.5:
            image = torch.flip(image, dims=[1]); mask = torch.flip(mask, dims=[0])
        k = random.randint(0, 3)
        return torch.rot90(image, k, dims=[1, 2]), torch.rot90(mask, k, dims=[0, 1])

    def __getitem__(self, idx):
        if len(self.items[idx]) == 3:
            case_dir, slice_idx, no_mask = self.items[idx]
        else:
            case_dir, slice_idx = self.items[idx]
            no_mask = False
        
        sample = load_case(case_dir)
        mods = sample["modalities"]
        image = np.stack([mods["flair"], mods["t1"], mods["t1ce"], mods["t2"]], axis=0)[..., slice_idx, :, :]
        mask = sample["mask"]
        mask2d = np.zeros((image.shape[-2], image.shape[-1]), np.float32) if mask is None else label_map(mask[slice_idx], self.label_mode)
        image_t = torch.from_numpy(image.copy()).float()
        mask_t = torch.from_numpy(mask2d.copy())
        if image_t.shape[-2:] != (self.image_size, self.image_size):
            image_t = F.interpolate(image_t.unsqueeze(0), size=(self.image_size, self.image_size), mode="bilinear", align_corners=False).squeeze(0)
            mask_t = F.interpolate(mask_t[None, None].float(), size=(self.image_size, self.image_size), mode="nearest").squeeze()
        if self.training and self.augment:
            image_t, mask_t = self._augment(image_t, mask_t)
        if self.label_mode == "whole_tumor":
            mask_t = mask_t.unsqueeze(0).float()
        else:
            mask_t = mask_t.long()
        return image_t, mask_t, case_dir.name, slice_idx

def build_loaders(config):
    train_cases = summary["train_cases"]
    val_cases = summary["val_cases"]
    test_cases = summary["test_cases"]
    official_val_cases = summary["official_val_cases"]
    
    if config["USE_DEBUG_SUBSET"]:
        train_cases = train_cases[:config["DEBUG_NUM_CASES"]]
        val_cases = val_cases[:config["DEBUG_NUM_CASES"]]
        test_cases = test_cases[:config["DEBUG_NUM_CASES"]]
        official_val_cases = official_val_cases[:config["DEBUG_NUM_CASES"]]
    
    train_ds = BratsSliceDataset(train_cases, config["image_size"], config["label_mode"], True, config["debug_max_slices_per_case"] if config["USE_DEBUG_SUBSET"] else None, True, allow_missing_seg=False)
    val_ds = BratsSliceDataset(val_cases, config["image_size"], config["label_mode"], False, 1 if config["USE_DEBUG_SUBSET"] else None, False, allow_missing_seg=False)
    test_ds = BratsSliceDataset(test_cases, config["image_size"], config["label_mode"], False, 1 if config["USE_DEBUG_SUBSET"] else None, False, allow_missing_seg=False)
    official_val_ds = BratsSliceDataset(official_val_cases, config["image_size"], config["label_mode"], False, 1 if config["USE_DEBUG_SUBSET"] else None, False, allow_missing_seg=True)
    
    train_loader = DataLoader(train_ds, batch_size=config["batch_size"], shuffle=True, num_workers=config["num_workers"])
    val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=config["num_workers"])
    test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=config["num_workers"])
    official_val_loader = DataLoader(official_val_ds, batch_size=1, shuffle=False, num_workers=config["num_workers"])
    
    return train_loader, val_loader, test_loader, official_val_loader

train_loader, val_loader, test_loader, official_val_loader = build_loaders(CONFIG)
print(f"\nDataloaders created:")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")
print(f"  Test batches: {len(test_loader)}")
print(f"  Official val batches (inference-only): {len(official_val_loader)}")

if len(train_loader.dataset) == 0:
    print("\n[INFO] No training data available.")

## Verification: Labeled Training Case

In [ ]:
def show_sample(sample):
    image_t, mask_t, case_id, slice_idx = sample
    image, mask = image_t.numpy(), mask_t.numpy()
    fig, axes = plt.subplots(1, 6, figsize=(18, 4))
    for i, title in enumerate(['FLAIR', 'T1', 'T1ce', 'T2']):
        axes[i].imshow(image[i], cmap='gray')
        axes[i].set_title(title)
        axes[i].axis('off')
    axes[4].imshow(mask.squeeze(), cmap='magma')
    axes[4].set_title('Mask')
    axes[4].axis('off')
    axes[5].imshow(image[0], cmap='gray')
    axes[5].imshow(mask.squeeze(), cmap='Reds', alpha=0.35)
    axes[5].set_title('Overlay')
    axes[5].axis('off')
    plt.suptitle(case_id + ' | slice ' + str(slice_idx))
    plt.tight_layout()
    plt.show()

if len(train_loader.dataset) > 0:
    sample = train_loader.dataset[0]
    image, mask, case_id, slice_idx = sample
    print('Training case verification:')
    print('  Case:', case_id)
    print('  Image shape:', image.shape)
    print('  Mask shape:', mask.shape)
    print('  OK - Labeled training case loaded successfully')
    if CONFIG["USE_DEBUG_SUBSET"]:
        print('  Sample plot skipped in startup smoke test; call show_sample(...) manually if needed.')

## Model Architecture: ExUTransLite

Dual-encoder with U-Net + ViT-style attention branch, SE-MHA, DAE, contextual self-attention, bivariate fusion.

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.net(x)

class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(nn.MaxPool2d(2), ConvBlock(in_ch, out_ch))
    def forward(self, x): return self.net(x)

class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, 2, 2)
        self.conv = ConvBlock(out_ch + skip_ch, out_ch)
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))

class PatchEmbed(nn.Module):
    def __init__(self, in_ch, embed_dim, patch_size):
        super().__init__()
        self.proj = nn.Conv2d(in_ch, embed_dim, patch_size, patch_size)
    def forward(self, x):
        x = self.proj(x)
        b, c, h, w = x.shape
        return x.flatten(2).transpose(1, 2), (h, w)

class SEMHABlock(nn.Module):
    def __init__(self, embed_dim, num_heads=4, mlp_ratio=4.0):
        super().__init__()
        self.n1 = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.eps = nn.Sequential(nn.Linear(embed_dim, embed_dim), nn.Tanh())
        self.n2 = nn.LayerNorm(embed_dim)
        hidden = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(nn.Linear(embed_dim, hidden), nn.GELU(), nn.Linear(hidden, embed_dim))
        self.attn_map = None
    def forward(self, x):
        x1 = self.n1(x)
        attn_out, attn_w = self.attn(x1, x1, x1, need_weights=True, average_attn_weights=False)
        x = x + attn_out + 0.1 * self.eps(x1)
        x = x + self.mlp(self.n2(x))
        self.attn_map = attn_w.mean(dim=1)
        return x

class DAE(nn.Module):
    def __init__(self, embed_dim, attr_dim=4):
        super().__init__()
        self.token_score = nn.Linear(embed_dim, 1)
        self.attr_head = nn.Sequential(nn.LayerNorm(embed_dim), nn.Linear(embed_dim, attr_dim))
    def forward(self, tokens, hw):
        attr_logits = self.attr_head(tokens.mean(dim=1))
        attr_map = torch.sigmoid(self.token_score(tokens)).transpose(1, 2).reshape(tokens.shape[0], 1, hw[0], hw[1])
        return attr_logits, attr_map

class ContextualSelfAttention(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.spatial = nn.Conv2d(2, 1, 7, padding=3)
        hidden = max(channels // 4, 8)
        self.channel = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(channels, hidden, 1), nn.ReLU(inplace=True), nn.Conv2d(hidden, channels, 1), nn.Sigmoid())
    def forward(self, x):
        avg, mx = x.mean(1, keepdim=True), x.amax(1, keepdim=True)
        spatial = torch.sigmoid(self.spatial(torch.cat([avg, mx], dim=1)))
        ch = self.channel(x)
        return x * spatial * ch, spatial, ch

class BivariateFusion(nn.Module):
    def __init__(self, local_ch, global_ch):
        super().__init__()
        self.local = nn.Conv2d(local_ch, local_ch, 1)
        self.global_ = nn.Conv2d(global_ch, local_ch, 1)
        self.spatial = nn.Conv2d(2, 1, 7, padding=3)
        hidden = max(local_ch // 4, 8)
        self.channel = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(local_ch, hidden, 1), nn.ReLU(inplace=True), nn.Conv2d(hidden, local_ch, 1), nn.Sigmoid())
    def forward(self, local_feat, global_feat):
        local_feat, global_feat = self.local(local_feat), self.global_(global_feat)
        fused = local_feat + global_feat
        spatial = torch.sigmoid(self.spatial(torch.cat([fused.mean(1, keepdim=True), fused.amax(1, keepdim=True)], dim=1)))
        ch = self.channel(fused)
        return fused * spatial * ch + local_feat, spatial, ch

class ExUTransLite(nn.Module):
    def __init__(self, in_ch=4, out_ch=1, patch_size=16, embed_dim=128):
        super().__init__()
        self.enc1, self.enc2, self.enc3, self.enc4 = ConvBlock(in_ch, 32), DownBlock(32, 64), DownBlock(64, 128), DownBlock(128, 256)
        self.patch = PatchEmbed(in_ch, embed_dim, patch_size)
        self.sem1, self.sem2 = SEMHABlock(embed_dim), SEMHABlock(embed_dim)
        self.dae = DAE(embed_dim)
        self.glob = nn.Conv2d(embed_dim, 256, 1)
        self.fuse = BivariateFusion(256, 256)
        self.ctx = ContextualSelfAttention(256)
        self.up3, self.up2, self.up1 = UpBlock(256, 128, 128), UpBlock(128, 64, 64), UpBlock(64, 32, 32)
        self.head = nn.Conv2d(32, out_ch, 1)
    def forward(self, x):
        s1 = self.enc1(x)
        s2 = self.enc2(s1)
        s3 = self.enc3(s2)
        bottleneck = self.enc4(s3)
        tokens, hw = self.patch(x)
        tokens = self.sem2(self.sem1(tokens))
        attr_logits, attr_map = self.dae(tokens, hw)
        global_map = tokens.transpose(1, 2).reshape(x.shape[0], -1, hw[0], hw[1])
        global_map = F.interpolate(global_map, size=bottleneck.shape[-2:], mode="bilinear", align_corners=False)
        global_map = self.glob(global_map)
        fused, fusion_spatial, fusion_channel = self.fuse(bottleneck, global_map)
        fused, ctx_spatial, ctx_channel = self.ctx(fused)
        d3 = self.up3(fused, s3)
        d2 = self.up2(d3, s2)
        d1 = self.up1(d2, s1)
        out = self.head(d1)
        return out, {"attr_logits": attr_logits, "attr_map": F.interpolate(attr_map, size=x.shape[-2:], mode="bilinear", align_corners=False), "attention_map": self.sem2.attn_map, "fusion_spatial": fusion_spatial, "fusion_channel": fusion_channel, "context_spatial": ctx_spatial, "context_channel": ctx_channel}

model = ExUTransLite(in_ch=4, out_ch=(1 if CONFIG["label_mode"] == "whole_tumor" else 4), patch_size=CONFIG["patch_size"]).to(DEVICE)
print(f"Model: {model.__class__.__name__}")

optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])

if len(train_loader.dataset) > 0:
    images, masks, case_id, slice_idx = next(iter(train_loader))
    images = images[:1].to(DEVICE)
    with torch.no_grad():
        logits, aux = model(images)
    print(f"\nModel forward pass verification:")
    print(f"  Input shape: {tuple(images.shape)} (B, C, H, W)")
    print(f"  Logits shape: {tuple(logits.shape)}")
    print(f"  Attr map shape: {tuple(aux['attr_map'].shape)}")
    print(f"  OK - Model forward pass works")

## Loss Functions and Metrics

TGOF (Trait-Guided Optimization Function): BCE + Dice + Alignment + Boundary-Aware Loss

In [ ]:
def dice_loss_binary(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)
    targets = targets.float()
    dims = tuple(range(1, probs.ndim))
    inter = (probs * targets).sum(dim=dims)
    denom = probs.sum(dim=dims) + targets.sum(dim=dims)
    return 1.0 - ((2 * inter + eps) / (denom + eps)).mean()

def multiclass_dice_loss(logits, targets, eps=1e-6):
    probs = torch.softmax(logits, dim=1)
    one_hot = F.one_hot(targets.long(), num_classes=probs.shape[1]).permute(0, 3, 1, 2).float()
    inter = (probs * one_hot).sum(dim=(0, 2, 3))
    denom = probs.sum(dim=(0, 2, 3)) + one_hot.sum(dim=(0, 2, 3))
    return 1.0 - ((2 * inter + eps) / (denom + eps)).mean()

def trait_guided_alignment_loss(attr_map, mask, eps=1e-6):
    """Align attention maps with tumor regions"""
    mask_normalized = torch.sigmoid(mask.float())
    return F.mse_loss(attr_map, mask_normalized)

def boundary_aware_loss(logits, targets, eps=1e-6):
    """Penalize gradient differences at boundaries"""
    pred = torch.sigmoid(logits)
    targets_f = targets.float()
    pred_grad_x = pred[:, :, 1:, :] - pred[:, :, :-1, :]
    pred_grad_y = pred[:, :, :, 1:] - pred[:, :, :, :-1]
    tgt_grad_x = targets_f[:, :, 1:, :] - targets_f[:, :, :-1, :]
    tgt_grad_y = targets_f[:, :, :, 1:] - targets_f[:, :, :, :-1]
    loss_x = F.mse_loss(pred_grad_x, tgt_grad_x)
    loss_y = F.mse_loss(pred_grad_y, tgt_grad_y)
    return (loss_x + loss_y) / 2

def segmentation_loss(logits, target, aux, label_mode="whole_tumor", attr_weight=0.1, align_weight=0.05, boundary_weight=0.05):
    """Trait-Guided Optimization Function (TGOF)"""
    if label_mode == "whole_tumor":
        bce = F.binary_cross_entropy_with_logits(logits, target.float())
        dice = dice_loss_binary(logits, target)
        align = trait_guided_alignment_loss(aux["attr_map"], target)
        boundary = boundary_aware_loss(logits, target)
        total_loss = bce + dice + attr_weight * align + align_weight * align + boundary_weight * boundary
        return total_loss, {"bce": bce.detach(), "dice": dice.detach(), "attr": align.detach(), "boundary": boundary.detach()}
    
    ce = F.cross_entropy(logits, target.long())
    dice = multiclass_dice_loss(logits, target.long())
    whole = (target > 0).float().unsqueeze(1)
    attr = F.binary_cross_entropy(aux["attr_map"], whole)
    align = trait_guided_alignment_loss(aux["attr_map"], whole)
    boundary = boundary_aware_loss(logits, whole)
    total_loss = ce + dice + attr_weight * attr + align_weight * align + boundary_weight * boundary
    return total_loss, {"ce": ce.detach(), "dice": dice.detach(), "attr": attr.detach(), "align": align.detach(), "boundary": boundary.detach()}

def to_pred(logits, label_mode="whole_tumor"):
    return (torch.sigmoid(logits) > 0.5).long() if label_mode == "whole_tumor" else torch.argmax(torch.softmax(logits, dim=1), dim=1, keepdim=True)

def dice_score(pred, target, eps=1e-6):
    pred, target = pred.astype(bool), target.astype(bool)
    inter = np.logical_and(pred, target).sum()
    return float((2 * inter + eps) / (pred.sum() + target.sum() + eps))

def iou_score(pred, target, eps=1e-6):
    pred, target = pred.astype(bool), target.astype(bool)
    inter = np.logical_and(pred, target).sum()
    union = np.logical_or(pred, target).sum()
    return float((inter + eps) / (union + eps))

def precision_score_np(pred, target, eps=1e-6):
    pred, target = pred.astype(bool), target.astype(bool)
    tp = np.logical_and(pred, target).sum()
    fp = np.logical_and(pred, np.logical_not(target)).sum()
    return float((tp + eps) / (tp + fp + eps))

def recall_score_np(pred, target, eps=1e-6):
    pred, target = pred.astype(bool), target.astype(bool)
    tp = np.logical_and(pred, target).sum()
    fn = np.logical_and(np.logical_not(pred), target).sum()
    return float((tp + eps) / (tp + fn + eps))

def f1_score_np(pred, target, eps=1e-6):
    p, r = precision_score_np(pred, target, eps), recall_score_np(pred, target, eps)
    return float((2 * p * r + eps) / (p + r + eps))

def hd95(pred, target):
    pred, target = pred.astype(bool), target.astype(bool)
    if pred.sum() == 0 or target.sum() == 0:
        return float("nan")
    pred_surf = pred ^ binary_erosion(pred)
    tgt_surf = target ^ binary_erosion(target)
    dist_a = distance_transform_edt(~target)[pred_surf]
    dist_b = distance_transform_edt(~pred)[tgt_surf]
    return float(np.percentile(np.concatenate([dist_a, dist_b]), 95)) if dist_a.size and dist_b.size else float("nan")

print("Loss functions and metrics defined with TGOF")
print("  - Pixel-level: BCE + Dice")
print("  - Alignment: Attribute map alignment")
print("  - Boundary: Edge-aware regularization")

## Training and Evaluation Loops

In [ ]:
def train_one_epoch(model, loader, optimizer, label_mode, attr_weight=0.1, align_weight=0.05, boundary_weight=0.05):
    model.train()
    total = 0.0
    loss_components = {"bce": 0, "dice": 0, "attr": 0, "align": 0, "boundary": 0}
    for images, masks, _, _ in tqdm(loader, desc="train", leave=False):
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits, aux = model(images)
        loss, loss_dict = segmentation_loss(logits, masks, aux, label_mode, attr_weight, align_weight, boundary_weight)
        loss.backward()
        optimizer.step()
        total += float(loss.item())
        for k in loss_components:
            if k in loss_dict:
                loss_components[k] += float(loss_dict[k].item())
    return total / max(len(loader), 1), {k: v / max(len(loader), 1) for k, v in loss_components.items()}

@torch.no_grad()
def evaluate(model, loader, label_mode):
    model.eval()
    scores = {"dice": [], "iou": [], "precision": [], "recall": [], "f1": [], "hd95": []}
    samples = []
    all_metrics = {}
    
    for images, masks, case_id, slice_idx in tqdm(loader, desc="eval", leave=False):
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        logits, aux = model(images)
        pred = to_pred(logits, label_mode).cpu().numpy().squeeze()
        target = masks.cpu().numpy().squeeze()
        if label_mode != "whole_tumor":
            target = (target > 0).astype(np.int64)
        
        dice = dice_score(pred, target)
        iou = iou_score(pred, target)
        precision = precision_score_np(pred, target)
        recall = recall_score_np(pred, target)
        f1 = f1_score_np(pred, target)
        hd = hd95(pred, target)
        
        scores["dice"].append(dice)
        scores["iou"].append(iou)
        scores["precision"].append(precision)
        scores["recall"].append(recall)
        scores["f1"].append(f1)
        scores["hd95"].append(hd)
        
        case_name = case_id[0] if isinstance(case_id, tuple) else case_id
        case_key = f"{case_name}_slice_{slice_idx[0].item() if isinstance(slice_idx, torch.Tensor) else slice_idx}"
        all_metrics[case_key] = {"dice": dice, "iou": iou, "precision": precision, "recall": recall, "f1": f1, "hd95": hd}
        samples.append((images.cpu(), masks.cpu(), pred, case_id, slice_idx, aux))
    
    return {k: float(np.nanmean(v)) for k, v in scores.items()}, samples, all_metrics

@torch.no_grad()
def show_prediction(sample_tuple, model, label_mode="whole_tumor"):
    images, masks, case_id, slice_idx = sample_tuple
    images = images.unsqueeze(0).to(DEVICE)
    logits, aux = model(images)
    pred = to_pred(logits, label_mode).cpu().numpy()[0]
    image = images.cpu().numpy()[0]
    mask = masks.numpy()[0]
    if label_mode != "whole_tumor":
        mask = (mask > 0).astype(np.float32)
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].imshow(image[0], cmap="gray")
    axes[0].set_title("FLAIR")
    axes[1].imshow(mask.squeeze(), cmap="magma")
    axes[1].set_title("Ground Truth")
    axes[2].imshow(pred.squeeze(), cmap="magma")
    axes[2].set_title("Prediction")
    axes[3].imshow(image[0], cmap="gray")
    axes[3].imshow(pred.squeeze(), cmap="Reds", alpha=0.35)
    axes[3].set_title("Overlay")
    for ax in axes:
        ax.axis("off")
    plt.suptitle(f"{case_id} | slice {slice_idx}")
    plt.tight_layout()
    plt.show()
    return pred, aux

if len(val_loader.dataset) > 0:
    print("Prediction visualization helper is ready; call show_prediction(...) manually if needed.")

## Output Export Functions

In [ ]:
import os
from pathlib import Path
import pandas as pd

OUTPUT_DIR = Path("outputs")
METRICS_DIR = OUTPUT_DIR / "metrics"
FIGURES_DIR = OUTPUT_DIR / "figures"
PREDICTIONS_DIR = OUTPUT_DIR / "predictions"
ATTENTION_DIR = OUTPUT_DIR / "attention_maps"

for d in [METRICS_DIR, FIGURES_DIR, PREDICTIONS_DIR, ATTENTION_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Output directories created:")
for d in [METRICS_DIR, FIGURES_DIR, PREDICTIONS_DIR, ATTENTION_DIR]:
    print(f"  {d}")

def export_metrics_table(all_metrics, dataset_type="validation"):
    """Export metrics table as CSV"""
    metrics_list = [{"case_id": case_id, **metrics} for case_id, metrics in all_metrics.items()]
    df = pd.DataFrame(metrics_list)
    csv_path = METRICS_DIR / f"metrics_{dataset_type}.csv"
    df.to_csv(csv_path, index=False)
    print(f"Metrics saved to: {csv_path}")
    return df

def export_comparison_chart(metrics_list):
    """Export metrics comparison figure"""
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    fig.suptitle("ExU-Trans Performance Metrics", fontsize=16, fontweight='bold')
    
    metrics_to_plot = [
        ("Dice", "dice", (0, 0)),
        ("IoU", "iou", (0, 1)),
        ("HD95", "hd95", (0, 2)),
        ("Precision", "precision", (1, 0)),
        ("Recall", "recall", (1, 1)),
        ("F1", "f1", (1, 2))
    ]
    
    for title, metric, (row, col) in metrics_to_plot:
        ax = axes[row, col]
        values = [m[metric] for m in metrics_list]
        ax.hist(values, bins=15, color='steelblue', edgecolor='black', alpha=0.7)
        ax.set_title(f"{title} (mean: {np.nanmean(values):.4f})")
        ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    fig_path = FIGURES_DIR / "metrics_comparison.png"
    plt.savefig(fig_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Figure saved to: {fig_path}")

print("Export functions defined")

## Main Training Block

Full training: 100 epochs when USE_DEBUG_SUBSET=False

In [ ]:
if not CONFIG["USE_DEBUG_SUBSET"] and len(train_loader.dataset) > 0:
    print(f"\nStarting training: {CONFIG['epochs']} epochs")
    
    best_val_dice = 0.0
    
    for epoch in range(CONFIG["epochs"]):
        tr_loss, loss_components = train_one_epoch(model, train_loader, optimizer, CONFIG["label_mode"], 
                                                    attr_weight=CONFIG["attr_loss_weight"], 
                                                    align_weight=0.05, 
                                                    boundary_weight=0.05)
        
        if (epoch + 1) % 5 == 0 or epoch == 0:
            val_metrics, val_samples, val_case_metrics = evaluate(model, val_loader, CONFIG["label_mode"])
            
            if val_metrics["dice"] > best_val_dice:
                best_val_dice = val_metrics["dice"]
            
            print(f"Epoch {epoch+1}/{CONFIG['epochs']} - Train loss: {tr_loss:.4f} - Val Dice: {val_metrics['dice']:.4f}")
    
    print(f"\nTraining complete. Best Val Dice: {best_val_dice:.4f}")
else:
    print("\nTraining skipped (debug mode or no data).")

## Reproduction Summary

ExU-Trans BraTS2020 reproduction as a 2D slice-based approximation:
- Labeled training (369 cases) split: 70% train, 15% val, 15% test
- Multi-modal (FLAIR, T1, T1ce, T2) with z-score normalization
- Dual-encoder: U-Net + ViT with SE-MHA, DAE, contextual self-attention, bivariate fusion
- TGOF loss: BCE + Dice + alignment + boundary-aware
- Metrics: Dice, IoU, Precision, Recall, F1, HD95

## Future Work: 3D Extension

This is a 2D slice-based approximation. For full 3D reproduction:
1. Load full 3D volumes
2. Implement 3D patch extraction (64x64x64)
3. Replace 2D convs with 3D convs
4. Compute metrics on 3D volumes
5. Use mixed precision/distributed training for memory